# Relational Feature Experiments

This notebook is an artifact reader for the full relational run. The heavy work is implemented in `scripts/generate_relational_profiles.py` and `scripts/run_relational_feature_experiments.py` so the experiment can be rerun reproducibly without embedding long-running aggregation logic in notebook cells.

In [1]:
from pathlib import Path
import pandas as pd

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / 'reports').exists() else CWD.parent
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR

WindowsPath('C:/Users/erenb/Desktop/Summer2026/DataScience/projects/home-credit-default-risk/reports')

## Profile JSON Summary

In [2]:
print((REPORTS_DIR / 'relational_profile_json_summary.md').read_text(encoding='utf-8')[:5000])

# Relational Profile JSON Summary

## application_test

Highest missingness fields:
- `COMMONAREA_AVG`: missing 0.687, type Numeric
- `COMMONAREA_MEDI`: missing 0.687, type Numeric
- `COMMONAREA_MODE`: missing 0.687, type Numeric
- `NONLIVINGAPARTMENTS_AVG`: missing 0.684, type Numeric
- `NONLIVINGAPARTMENTS_MEDI`: missing 0.684, type Numeric
- `NONLIVINGAPARTMENTS_MODE`: missing 0.684, type Numeric
- `FONDKAPREMONT_MODE`: missing 0.673, type Text
- `LIVINGAPARTMENTS_MEDI`: missing 0.672, type Numeric

Highest distinct-rate fields:
- `SK_ID_CURR`: distinct-rate 1.000, n_distinct 48744
- `EXT_SOURCE_1`: distinct-rate 0.964, n_distinct 27207
- `EXT_SOURCE_2`: distinct-rate 0.798, n_distinct 38885
- `DAYS_BIRTH`: distinct-rate 0.318, n_distinct 15477
- `DAYS_REGISTRATION`: distinct-rate 0.259, n_distinct 12618
- `DAYS_EMPLOYED`: distinct-rate 0.161, n_distinct 7863
- `LIVINGAREA_MEDI`: distinct-rate 0.154, n_distinct 3885
- `AMT_ANNUITY`: distinct-rate 0.154, n_distinct 7491

## applicati

## Validation Results

In [3]:
results_df = pd.read_csv(REPORTS_DIR / 'relational_feature_experiment_results.csv')
results_df.drop(columns=['top_20_features']).sort_values('validation_roc_auc', ascending=False)

,feature_set,raw_feature_count,threshold,validation_roc_auc,validation_average_precision,validation_accuracy,validation_precision_class_1,validation_recall_class_1,validation_f1_class_1,groups
0,app_plus_bureau_plus_previous_application_plus...,908,0.687520,0.787245,0.272062,0.866814,0.280788,0.416163,0.335328,"bureau,previous_application,installments_payme..."
1,app_plus_bureau_plus_previous_application_plus...,754,0.670884,0.786036,0.269483,0.857424,0.267887,0.442095,0.333618,"bureau,previous_application,installments_payments"
2,app_plus_bureau_plus_previous_application_plus...,799,0.677015,0.785561,0.271133,0.860392,0.270335,0.429255,0.331744,"bureau,previous_application,installments_payme..."
3,app_plus_bureau_plus_previous_application,709,0.673541,0.779348,0.260155,0.856835,0.263765,0.431772,0.327478,"bureau,previous_application"
4,app_plus_installments_payments,173,0.639584,0.773751,0.255888,0.834600,0.237591,0.474824,0.316709,installments_payments
5,app_plus_previous_application,535,0.665006,0.772954,0.250844,0.850026,0.251930,0.435549,0.319218,previous_application
6,app_plus_bureau,302,0.668245,0.770063,0.256343,0.852486,0.253525,0.425478,0.317729,bureau
7,app_plus_POS_CASH_balance,173,0.670222,0.768145,0.252689,0.851957,0.252762,0.426234,0.317338,POS_CASH_balance
8,app_plus_credit_card_balance,237,0.660810,0.767240,0.248766,0.845047,0.243755,0.437311,0.313029,credit_card_balance
9,app_expanded_only,128,0.688844,0.762402,0.243919,0.861225,0.257637,0.382175,0.307786,NaN


## Final Holdout Metrics

In [4]:
final_metrics_df = pd.read_csv(REPORTS_DIR / 'relational_final_test_metrics.csv')
final_metrics_df

,feature_set,threshold_strategy,threshold,roc_auc,average_precision,accuracy,precision_class_1,recall_class_1,f1_class_1
0,app_plus_bureau_plus_previous_application_plus...,default_0.5,0.50000,0.790203,0.288818,0.738956,0.192184,0.697281,0.301319
1,app_plus_bureau_plus_previous_application_plus...,validation_selected,0.68752,0.790203,0.288818,0.869600,0.289514,0.423162,0.343806


## Final Confusion Matrix

In [5]:
pd.read_csv(REPORTS_DIR / 'relational_final_confusion_matrix.csv', index_col=0)

,predicted_0,predicted_1
actual_0,51382,5156
actual_1,2864,2101


## Final Feature Importance

In [6]:
importance_df = pd.read_csv(REPORTS_DIR / 'relational_final_feature_importance.csv')
importance_df.head(50)

,transformed_feature,gain_importance
0,numeric__EXT_SOURCE_MEAN,754268.248363
1,numeric__EXT_SOURCE_MIN,111120.871225
2,numeric__CREDIT_TERM_APPROX,67951.873411
3,numeric__EXT_SOURCE_MAX,59565.386303
4,numeric__BURO_BURO_DEBT_CREDIT_RATIO_MAX,56662.530704
5,numeric__INST_INST_LATE_PAYMENT_FLAG_MEAN,49238.302052
6,numeric__GOODS_CREDIT_RATIO,44594.020910
7,numeric__AGE_YEARS,36856.707182
8,numeric__PREV_DAYS_LAST_DUE_1ST_VERSION_MAX,32552.885391
9,numeric__PREV_PREV_CREDIT_APPLICATION_RATIO_MEAN,29993.906042


## Main Findings

The all-relational configuration won validation ROC-AUC. Installment behavior, bureau debt ratios, previous application refusal/amount ratios, POS completion, and credit-card utilization/drawing behavior added useful ranking signal on top of the application-only model. The run is still below 0.80 ROC-AUC, so the next move is tuning LightGBM and adding recent-window aggregates rather than changing only the classification threshold.